# Ethos AI Evaluator - Kaggle GPU Server

**Everything runs on Kaggle T4 GPUs:**
- Model loading (8-bit quantization)
- Inference/generation
- LoRA training

## Setup:
1. Settings → Accelerator → **GPU T4 x2**
2. Settings → Internet → **On**
3. Run all cells
4. Copy ngrok URL to your local `backend/.env`

In [ ]:
# Install all dependencies
!pip install -q fastapi uvicorn python-multipart pyngrok
!pip install -q peft>=0.7.0 transformers>=4.36.0 accelerate>=0.25.0
!pip install -q bitsandbytes>=0.41.0 datasets>=2.14.0

In [ ]:
# Setup ngrok authentication
from pyngrok import ngrok, conf
import getpass

print("Get your ngrok authtoken from: https://dashboard.ngrok.com/get-started/your-authtoken")
ngrok_token = getpass.getpass("Enter your ngrok authtoken: ")
conf.get_default().auth_token = ngrok_token
print("✅ ngrok configured")

In [ ]:
# Verify GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB")

In [ ]:
# Create inference server code
inference_server_code = '''
import asyncio
import json
import logging
import os
import time
import uuid
from datetime import datetime, timezone
from typing import Optional, Dict, Any

import torch
from fastapi import FastAPI, HTTPException, UploadFile, File
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("inference_server")

app = FastAPI(title="Ethos Inference Server - Kaggle")

# Global state
model = None
tokenizer = None
model_info = {}
_loading = False
_adapter_loaded = False
_adapter_path = None
_base_model_ref = None
training_jobs = {}

class LoadRequest(BaseModel):
    model_name: str
    load_8bit: bool = True

class GenerateRequest(BaseModel):
    prompt: str
    max_tokens: int = 512
    temperature: float = 0.7
    top_p: float = 0.9

class TrainRequest(BaseModel):
    base_model: str
    round_num: int = 1
    epochs: int = 3
    lr: float = 2e-4
    batch_size: int = 4

def _unload():
    global model, tokenizer, model_info, _adapter_loaded, _adapter_path, _base_model_ref
    if model is not None:
        del model
        model = None
    if tokenizer is not None:
        del tokenizer
        tokenizer = None
    _adapter_loaded = False
    _adapter_path = None
    _base_model_ref = None
    model_info = {}
    torch.cuda.empty_cache()
    logger.info("Model unloaded")

def _do_load(model_name: str, load_8bit: bool = True):
    global model, tokenizer, model_info, _loading, _adapter_loaded, _adapter_path, _base_model_ref
    _loading = True
    try:
        _unload()
        logger.info(f"Loading model: {model_name}")
        t0 = time.time()
        
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        if tokenizer.pad_token is None and tokenizer.eos_token:
            tokenizer.pad_token = tokenizer.eos_token
        
        config = AutoConfig.from_pretrained(model_name)
        auto_cls = AutoModelForCausalLM
        
        if torch.cuda.is_available() and load_8bit:
            from transformers import BitsAndBytesConfig
            bnb_config = BitsAndBytesConfig(load_in_8bit=True, llm_int8_threshold=6.0)
            model = auto_cls.from_pretrained(
                model_name,
                quantization_config=bnb_config,
                device_map="auto",
                low_cpu_mem_usage=True,
            )
            logger.info("Model loaded in 8-bit")
        else:
            model = auto_cls.from_pretrained(
                model_name,
                torch_dtype=torch.float16,
                device_map="auto",
                low_cpu_mem_usage=True,
            )
        
        model.eval()
        elapsed = time.time() - t0
        device = next(model.parameters()).device
        
        _adapter_loaded = False
        _adapter_path = None
        _base_model_ref = None
        
        model_info = {
            "model_name": model_name,
            "device": str(device),
            "load_time_seconds": round(elapsed, 1),
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            "quantization": "8bit" if load_8bit else "float16",
            "adapter": None,
        }
        logger.info(f"Model loaded in {elapsed:.1f}s")
    finally:
        _loading = False

@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "model_loaded": model is not None,
        "model_info": model_info,
        "gpu_available": torch.cuda.is_available(),
        "adapter_loaded": _adapter_loaded,
    }

@app.post("/load")
async def load_model(req: LoadRequest):
    if _loading:
        raise HTTPException(status_code=409, detail="Model is currently loading")
    try:
        _do_load(req.model_name, req.load_8bit)
        return {"status": "loaded", "model_info": model_info}
    except Exception as e:
        logger.error(f"Failed to load model: {e}")
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/unload")
async def unload_model():
    _unload()
    return {"status": "unloaded"}

@app.post("/generate")
async def generate(req: GenerateRequest):
    if model is None:
        raise HTTPException(status_code=400, detail="No model loaded")
    try:
        inputs = tokenizer(req.prompt, return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=req.max_tokens,
                temperature=req.temperature,
                top_p=req.top_p,
                do_sample=True,
                pad_token_id=tokenizer.pad_token_id,
            )
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        return {"response": response, "tokens": len(outputs[0])}
    except Exception as e:
        logger.error(f"Generation failed: {e}")
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/upload_patch")
async def upload_patch(file: UploadFile = File(...)):
    try:
        content = await file.read()
        patch_path = "/kaggle/working/ethics_patch.jsonl"
        with open(patch_path, "wb") as f:
            f.write(content)
        with open(patch_path, "r") as f:
            entries = len(f.readlines())
        logger.info(f"Patch uploaded: {entries} entries")
        return {"status": "uploaded", "path": patch_path, "entries": entries}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/train")
async def start_training(req: TrainRequest):
    job_id = str(uuid.uuid4())[:8]
    training_jobs[job_id] = {
        "status": "queued",
        "started_at": datetime.now(timezone.utc).isoformat(),
        "progress": {"epoch": 0, "loss": None},
        "result": None,
        "error": None,
    }
    logger.info(f"Training job {job_id} queued")
    asyncio.create_task(_run_training(job_id, req))
    return {"job_id": job_id, "status": "started"}

async def _run_training(job_id: str, req: TrainRequest):
    try:
        import train_lora
        output_dir = f"/kaggle/working/adapters/round_{req.round_num}"
        result = train_lora.train(
            base_model=req.base_model,
            dataset_path="/kaggle/working/ethics_patch.jsonl",
            output_dir=output_dir,
            epochs=req.epochs,
            lr=req.lr,
            batch_size=req.batch_size,
        )
        training_jobs[job_id]["status"] = "completed"
        training_jobs[job_id]["result"] = result
        logger.info(f"Training job {job_id} completed")
    except Exception as e:
        logger.error(f"Training job {job_id} failed: {e}")
        training_jobs[job_id]["status"] = "failed"
        training_jobs[job_id]["error"] = str(e)

@app.get("/train_status")
async def get_training_status():
    status_file = "/kaggle/working/train_status.json"
    if os.path.exists(status_file):
        with open(status_file, "r") as f:
            status = json.load(f)
    else:
        status = {"status": "idle", "epoch": 0, "loss": None}
    latest_job = None
    for job_id, job_data in training_jobs.items():
        if latest_job is None or job_data["started_at"] > latest_job["started_at"]:
            latest_job = job_data
            latest_job["job_id"] = job_id
    if latest_job:
        return {
            "status": latest_job["status"],
            "progress": status if latest_job["status"] == "training" else latest_job.get("progress", {}),
            "result": latest_job.get("result"),
            "error": latest_job.get("error"),
        }
    return {"status": "idle", "progress": {}, "result": None, "error": None}

@app.post("/load_adapter")
async def load_adapter(adapter_path: str):
    global model, _adapter_loaded, _adapter_path, _base_model_ref
    if model is None:
        raise HTTPException(status_code=400, detail="No base model loaded")
    try:
        from peft import PeftModel
        if _adapter_loaded:
            model = _base_model_ref
            _adapter_loaded = False
        if _base_model_ref is None:
            _base_model_ref = model
        logger.info(f"Loading adapter from {adapter_path}")
        model = PeftModel.from_pretrained(_base_model_ref, adapter_path)
        model.eval()
        _adapter_loaded = True
        _adapter_path = adapter_path
        model_info["adapter"] = adapter_path
        return {"status": "loaded", "adapter_path": adapter_path}
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

@app.post("/unload_adapter")
async def unload_adapter():
    global model, _adapter_loaded, _adapter_path
    if not _adapter_loaded:
        return {"status": "no_adapter_loaded"}
    model = _base_model_ref
    _adapter_loaded = False
    _adapter_path = None
    model_info["adapter"] = None
    return {"status": "unloaded"}
'''

with open('/kaggle/working/inference_server.py', 'w') as f:
    f.write(inference_server_code)

print("✅ inference_server.py created")

In [ ]:
# Create training module
train_lora_code = '''
import json
import logging
import os
import time
from datetime import datetime, timezone
from typing import Dict, Any

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("train_lora")

def train(
    base_model: str,
    dataset_path: str = "/kaggle/working/ethics_patch.jsonl",
    output_dir: str = "/kaggle/working/adapters/round_1",
    epochs: int = 3,
    lr: float = 2e-4,
    batch_size: int = 4,
    lora_r: int = 8,
    lora_alpha: int = 32,
) -> Dict[str, Any]:
    start_time = time.time()
    logger.info(f"Starting LoRA training: {base_model}")
    
    _update_status({"status": "loading_model", "epoch": 0, "loss": None})
    
    tokenizer = AutoTokenizer.from_pretrained(base_model)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    logger.info("Loading base model in 8-bit...")
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(load_in_8bit=True, llm_int8_threshold=6.0)
    model = AutoModelForCausalLM.from_pretrained(
        base_model,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    
    logger.info(f"Applying LoRA config: r={lora_r}, alpha={lora_alpha}")
    lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    logger.info(f"Loading dataset from {dataset_path}")
    _update_status({"status": "loading_dataset", "epoch": 0, "loss": None})
    dataset = load_dataset("json", data_files=dataset_path, split="train")
    
    def tokenize_function(examples):
        texts = [f"{p}\\n{c}" for p, c in zip(examples["prompt"], examples["completion"])]
        return tokenizer(texts, truncation=True, max_length=512, padding="max_length")
    
    tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        learning_rate=lr,
        logging_steps=1,
        save_strategy="epoch",
        save_total_limit=1,
        fp16=True,
        report_to="none",
    )
    
    class StatusCallback:
        def on_epoch_begin(self, args, state, control, **kwargs):
            _update_status({"status": "training", "epoch": int(state.epoch), "loss": None})
        def on_log(self, args, state, control, logs=None, **kwargs):
            if logs and "loss" in logs:
                _update_status({"status": "training", "epoch": int(state.epoch), "loss": round(logs["loss"], 4)})
    
    logger.info("Starting training...")
    _update_status({"status": "training", "epoch": 0, "loss": None})
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_dataset,
        callbacks=[StatusCallback()],
    )
    
    train_result = trainer.train()
    
    adapter_path = os.path.join(output_dir, "adapter")
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    
    duration = time.time() - start_time
    metrics = {
        "train_loss": round(train_result.training_loss, 4),
        "epochs": epochs,
        "dataset_size": len(dataset),
        "duration_seconds": round(duration, 1),
        "adapter_path": adapter_path,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }
    
    logger.info(f"Training complete: loss={metrics['train_loss']}, duration={duration:.1f}s")
    _update_status({"status": "completed", "epoch": epochs, "loss": metrics["train_loss"]})
    return metrics

def _update_status(status: Dict[str, Any]):
    status_file = "/kaggle/working/train_status.json"
    try:
        with open(status_file, "w") as f:
            json.dump(status, f)
    except Exception as e:
        pass
'''

with open('/kaggle/working/train_lora.py', 'w') as f:
    f.write(train_lora_code)

print("✅ train_lora.py created")

In [ ]:
# Start the inference server with ngrok tunnel
import sys
sys.path.append('/kaggle/working')

import threading
import uvicorn
from pyngrok import ngrok
import time

# Import the app
from inference_server import app

# Start uvicorn in background
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8080, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(5)

# Create ngrok tunnel
public_url = ngrok.connect(8080, bind_tls=True)

print("\n" + "="*80)
print("🚀 ETHOS INFERENCE SERVER RUNNING ON KAGGLE")
print("="*80)
print(f"\n📡 Public URL: {public_url}")
print(f"\n⚠️  COPY THIS URL TO YOUR LOCAL backend/.env FILE:")
print(f"\n   REMOTE_MODEL_URL={public_url}")
print("\n" + "="*80)
print("\n✅ Server Status:")
print("   - Model loading: Ready (call /load endpoint)")
print("   - Inference: Ready (call /generate endpoint)")
print("   - LoRA Training: Ready (call /train endpoint)")
print("\n💡 Keep this notebook running while testing.")
print("   Session will auto-stop after 12 hours.\n")
print("="*80)

In [ ]:
# Keep notebook alive
import time
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nShutting down...")
    ngrok.disconnect(public_url)
    print("Server stopped.")